# Clase 2 — Construye tu primera red neuronal desde cero

**Objetivos de hoy**
- Entender qué es un tensor y por qué TensorFlow se llama así.
- Definir a mano los pesos y sesgos de una red densa.
- Entrenarla con `tf.GradientTape` y comparar el resultado con la Keras de la clase anterior.

Consulta el [capítulo 1 del manual](../manual/01_fundamentos.md) para la explicación completa.

## 1. ¿Qué es un tensor?

Un tensor es un array multidimensional: un número suelto es un tensor de dimensión cero, una lista es de dimensión uno, una imagen en blanco y negro es de dimensión dos. TensorFlow representa así tanto tus datos como los pesos internos de la red.

In [ ]:
import tensorflow as tf

escalar = tf.constant(7)
vector = tf.constant([1, 2, 3])
matriz = tf.constant([[1, 2], [3, 4]])

print("Escalar (dimensión 0):", escalar.shape)
print("Vector (dimensión 1):", vector.shape)
print("Matriz (dimensión 2):", matriz.shape)

## 2. Carga los mismos datos que la clase anterior, en formato "aplanado"

In [ ]:
(x_entreno, y_entreno), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_entreno = (x_entreno / 255.0).reshape(-1, 784).astype("float32")
x_test = (x_test / 255.0).reshape(-1, 784).astype("float32")

y_entreno_onehot = tf.keras.utils.to_categorical(y_entreno, 10)
y_test_onehot = tf.keras.utils.to_categorical(y_test, 10)

print("Forma de x_entreno:", x_entreno.shape)

## 3. Define pesos y sesgos como variables entrenables

Nada de `Dense`: vas a crear tú mismo la matriz de pesos y el vector de sesgos.

In [ ]:
pesos = tf.Variable(tf.random.normal([784, 10], stddev=0.1))
sesgos = tf.Variable(tf.zeros([10]))

def prediccion(x):
    return tf.nn.softmax(tf.matmul(x, pesos) + sesgos)

## 4. Entrena con `tf.GradientTape`

La "cinta" graba las operaciones mientras se ejecutan, para poder calcular después el gradiente de cada peso: la dirección en la que hay que moverlo para reducir el error.

In [ ]:
optimizador = tf.keras.optimizers.Adam(learning_rate=0.01)
tamano_lote = 128

for epoca in range(3):
    for i in range(0, len(x_entreno), tamano_lote):
        x_lote = x_entreno[i:i + tamano_lote]
        y_lote = y_entreno_onehot[i:i + tamano_lote]
        with tf.GradientTape() as cinta:
            y_predicha = prediccion(x_lote)
            perdida = tf.keras.losses.categorical_crossentropy(y_lote, y_predicha)
        gradientes = cinta.gradient(perdida, [pesos, sesgos])
        optimizador.apply_gradients(zip(gradientes, [pesos, sesgos]))
    print(f"Época {epoca + 1} completada")

## 5. Evalúa tu red manual

In [ ]:
predicciones_test = prediccion(x_test)
aciertos = tf.equal(tf.argmax(predicciones_test, axis=1), tf.argmax(y_test_onehot, axis=1))
exactitud = tf.reduce_mean(tf.cast(aciertos, tf.float32))
print(f"Exactitud sobre el conjunto de test: {exactitud.numpy():.2%}")

## Resumen

- Una red de una sola capa densa, construida a mano, ya reconoce dígitos con bastante exactitud (aunque menos que la red de dos capas de Keras de la clase 1).
- `tf.Variable` guarda los pesos entrenables; `tf.GradientTape` calcula sus gradientes; el optimizador los actualiza.
- Esto es exactamente lo que Keras hace por ti cuando llamas a `fit`: ahora ya sabes qué hay dentro.

Continúa con el [notebook de práctica de esta clase](../notebooks_practica/clase02_practica_red_manual.ipynb).